In [1]:
!nvidia-smi

Sat Jun  6 13:17:21 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   37C    P8             12W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
!git clone --branch main https://github.com/omerzkan/eomt-anomaly-segmentation.git cloned_repo_feature_omer
%cd cloned_repo_feature_omer

Cloning into 'cloned_repo_feature_omer'...
remote: Enumerating objects: 591, done.
remote: Counting objects: 100% (23/23), done.
remote: Compressing objects: 100% (16/16), done.
remote: Total 591 (delta 10), reused 17 (delta 6), pack-reused 568 (from 1)
Receiving objects: 100% (591/591), 27.03 MiB | 9.94 MiB/s, done.
Resolving deltas: 100% (292/292), done.
/content/cloned_repo_feature_omer


In [4]:
!pip install -r eval/requirements.txt
!pip install -r eomt/requirements.txt

INFO: pip is looking at multiple versions of ood-metrics to determine which version is compatible with other requirements. This could take a while.
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 5.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 4.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 5.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 5.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 4.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 219.4/219.4 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.6/8.6 MB 154.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 83.4 MB/s eta 0:00:00

In [ ]:
# fine-tuning
%cd /content/cloned_repo_feature_omer/eomt

!python3 main.py fit \
  -c configs/dinov2/cityscapes/semantic/eomt_base_640.yaml \
  --model.ckpt_path /content/drive/MyDrive/FAIMDL/checkpoints/eomt_coco.bin \
  --model.load_ckpt_class_head False \
  --model.lr_mult 0.0 \
  --trainer.precision 16-mixed \
  --trainer.max_epochs 20 \
  --trainer.devices 1 \
  --data.batch_size 4 \
  --data.path /content/drive/MyDrive/FAIMDL/data \
  --data.img_size "[640,640]" \
  --model.network.num_q 200

/content/cloned_repo_feature_omer/eomt
2026-05-23 11:36:39.418258: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-05-23 11:36:39.483535: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Seed set to 0
INFO:root:Loaded 195 keys
Using 16bit Automatic Mixed Precision (AMP)
Using default `ModelCheckpoint`. Consider installing `litmodels` package to enable `LitModelCheckpoint` for automatic upload to the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available

In [ ]:
# Find and save the checkpoint
import glob, shutil

ckpt_files = glob.glob("/content/cloned_repo_feature_omer/eomt/**/checkpoints/*.ckpt", recursive=True)
print("Found:", ckpt_files)

# Copy the best checkpoint to Drive
if ckpt_files:
    # Extract state_dict and save as .bin
    import torch
    ckpt = torch.load(ckpt_files[0], map_location="cpu", weights_only=False)
    state_dict = ckpt.get("state_dict", ckpt)
    torch.save({"state_dict": state_dict}, "/content/drive/MyDrive/FAIMDL/checkpoints/coco_eomt_finetuned_on_cityscapes.bin")
    print("Saved to Drive!")

Found: ['/content/cloned_repo_feature_omer/eomt/eomt/z97cnktr/checkpoints/epoch=19-step=14860.ckpt']
Saved to Drive!


In [1]:
%cd /content/cloned_repo_feature_omer
!git pull

import os

current_dir = os.getcwd()
print("current dir is: ")
if not current_dir.endswith('cloned_repo_feature_omer'):
    %cd /content/cloned_repo_feature_omer
    print(current_dir)
else:
  print(current_dir)

/content/cloned_repo_feature_omer
Already up to date.
current dir is: 
/content/cloned_repo_feature_omer


In [2]:
%cd /content/cloned_repo_feature_omer
!python3 step5/eval_finetuned.py

/content/cloned_repo_feature_omer
Val set size: 500 images
2026-06-06 13:23:43.697931: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-06-06 13:23:43.763173: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
model.safetensors: 100% 346M/346M [00:02<00:00, 139MB/s]
/usr/local/lib/python3.12/dist-packages/lightning/pytorch/utilities/parsing.py:209: Attribute 'network' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['networ